In [ ]:
import numpy as np
import matplotlib.pyplot as plt

output_dir = "entrega/imagens/"

# --------------------------------------------------------------------------
# System and Simulation Configuration
# --------------------------------------------------------------------------

# True system parameters
a1_true = 0.707
b1_true = 0.293
theta_true = np.array([a1_true, b1_true])

# Noise covariance
noise_variance = 3.0

# Simulation parameters
N = 1000  # Number of data samples
num_realizations = 100  # Number of different noise realizations


# --------------------------------------------------------------------------
# Function to Simulate the ARX System
# --------------------------------------------------------------------------
def simulate_arx_system(N, theta, noise_var):
    """
    Simulates the ARX system y(k) = a1*y(k-1) + b1*u(k-1) + eta(k) + 0.5 * eta(k - 1).

    Args:
        N (int): Number of samples.
        theta (np.array): True parameter vector [a1, b1].
        noise_var (float): Variance of the white noise eta(k).

    Returns:
        tuple: (u, y) input and output signals.
    """
    a1, b1 = theta[0], theta[1]

    # Generate a persistently exciting input signal (white noise)
    u = np.random.randn(N)

    # Generate white noise with specified variance
    eta = np.sqrt(noise_var) * np.random.randn(N)

    # Initialize output vector
    y = np.zeros(N)

    # Simulate the system dynamics
    for k in range(1, N):
        y[k] = a1 * y[k - 1] + b1 * u[k - 1] + eta[k] + 0.5 * eta[k - 1]

    return u, y


# --------------------------------------------------------------------------
# Recursive Estimator Implementations
# --------------------------------------------------------------------------
def estimate_gradient_like(y, u, K_bar):
    """
    Estimates parameters using the gradient-like recursive estimator.
    Update rule: theta_k = theta_{k-1} + K_bar * psi_k * error_k

    Args:
        y (np.array): Output signal.
        u (np.array): Input signal.
        K_bar (float): Fixed gain factor.

    Returns:
        np.array: Final estimated parameter vector.
    """
    # Initial parameter estimates
    theta_est = np.zeros(2)
    N = len(y)

    for k in range(1, N):
        # Form the regressor vector Psi_k = [y(k-1), u(k-1)]^T
        psi = np.array([y[k - 1], u[k - 1]])

        # Prediction error
        error = y[k] - psi.T @ theta_est

        # Gain vector K_k = K_bar * Psi_k
        Kk = K_bar * psi

        # Update parameter estimates
        theta_est = theta_est + Kk * error

    return theta_est


def estimate_gauss_newton_like(y, u):
    """
    Estimates parameters using the Gauss-Newton-like recursive estimator.
    Update rule: theta_k = theta_{k-1} + (psi_k^T * psi_k)^-1 * psi_k * error_k

    Args:
        y (np.array): Output signal.
        u (np.array): Input signal.

    Returns:
        np.array: Final estimated parameter vector.
    """
    # Initial parameter estimates
    theta_est = np.zeros(2)
    N = len(y)

    for k in range(1, N):
        # Form the regressor vector Psi_k = [y(k-1), u(k-1)]^T
        psi = np.array([y[k - 1], u[k - 1]])

        # Calculate psi.T * psi, add epsilon for numerical stability
        psi_norm_sq = psi.T @ psi
        if psi_norm_sq < 1e-9:
            continue

        # Prediction error
        error = y[k] - psi.T @ theta_est

        # Gain vector K_k = (Psi_k^T * Psi_k)^-1 * Psi_k
        Kk = (1 / psi_norm_sq) * psi

        # Update parameter estimates
        theta_est = theta_est + Kk * error

    return theta_est


# Store final estimates from all realizations
grad_estimates = np.zeros((num_realizations, 2))
gn_estimates = np.zeros((num_realizations, 2))

# A fixed K that does not lead to a very large covariance
# This choice may require tuning. A small value is generally safer.
K_bar_fixed = 0.01

print(f"Executando {num_realizations} simulacoes...")

for i in range(num_realizations):
    # Simulate the system for a new noise realization
    u, y = simulate_arx_system(N, theta_true, noise_variance)

    # a) Estimate parameters with both methods
    grad_estimates[i, :] = estimate_gradient_like(y, u, K_bar_fixed)
    gn_estimates[i, :] = estimate_gauss_newton_like(y, u)

print("Simulacoes concluidas. Gerando histogramas...")

# --- Combined Histogram Figure for All Estimators ---
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Histogramas dos Estimadores Recursivos")

# Top row: Gradient-like estimator
axs[0, 0].hist(grad_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs[0, 0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs[0, 0].axvline(
    np.mean(grad_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(grad_estimates[:, 0]):.4f}",
)
axs[0, 0].set_title("Gradiente: $\\hat{a}_1$")
axs[0, 0].set_xlabel("Valor Estimado de $a_1$")
axs[0, 0].set_ylabel("Frequencia")
axs[0, 0].legend()
axs[0, 0].grid(True)

axs[0, 1].hist(grad_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs[0, 1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs[0, 1].axvline(
    np.mean(grad_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(grad_estimates[:, 1]):.4f}",
)
axs[0, 1].set_title("Gradiente: $\\hat{b}_1$")
axs[0, 1].set_xlabel("Valor Estimado de $b_1$")
axs[0, 1].set_ylabel("Frequencia")
axs[0, 1].legend()
axs[0, 1].grid(True)

# Bottom row: Gauss-Newton-like estimator
axs[1, 0].hist(gn_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs[1, 0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs[1, 0].axvline(
    np.mean(gn_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(gn_estimates[:, 0]):.4f}",
)
axs[1, 0].set_title("Gauss-Newton: $\\hat{a}_1$")
axs[1, 0].set_xlabel("Valor Estimado de $a_1$")
axs[1, 0].set_ylabel("Frequencia")
axs[1, 0].legend()
axs[1, 0].grid(True)

axs[1, 1].hist(gn_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs[1, 1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs[1, 1].axvline(
    np.mean(gn_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(gn_estimates[:, 1]):.4f}",
)
axs[1, 1].set_title("Gauss-Newton: $\\hat{b}_1$")
axs[1, 1].set_xlabel("Valor Estimado de $b_1$")
axs[1, 1].set_ylabel("Frequencia")
axs[1, 1].legend()
axs[1, 1].grid(True)

fig.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1a_histograma.png")

In [ ]:
# --------------------------------------------------------------------------
# Stochastic Approximation Estimator Implementations
# --------------------------------------------------------------------------
def estimate_stochastic_approx_grad(y, u, K_bar, alpha):
    """
    Estimates parameters using the stochastic approximation (gradient-like) estimator.
    Update rule: theta_k = theta_{k-1} + K_bar * k**(-alpha) * psi_k * error_k
    """
    theta_est = np.zeros(2)
    N = len(y)

    for k in range(1, N):
        psi = np.array([y[k - 1], u[k - 1]])
        error = y[k] - psi.T @ theta_est

        # Gain vector with time-decaying term
        Kk = K_bar * (k ** (-alpha)) * psi

        theta_est = theta_est + Kk * error

    return theta_est


def estimate_stochastic_approx_gn(y, u, alpha):
    """
    Estimates parameters using the stochastic approximation (Gauss-Newton-like) estimator.
    Update rule: theta_k = theta_{k-1} + k**(-alpha) * (psi_k^T*psi_k)^-1*psi_k*error_k
    """
    theta_est = np.zeros(2)
    N = len(y)

    for k in range(1, N):
        psi = np.array([y[k - 1], u[k - 1]])
        psi_norm_sq = psi.T @ psi
        if psi_norm_sq < 1e-9:
            continue

        error = y[k] - psi.T @ theta_est

        # Gain vector with time-decaying term
        Kk = (k ** (-alpha)) * (1 / psi_norm_sq) * psi

        theta_est = theta_est + Kk * error

    return theta_est


# Store final estimates
sa_grad_estimates = np.zeros((num_realizations, 2))
sa_gn_estimates = np.zeros((num_realizations, 2))

# Use the same K_bar as before and an alpha within the suggested range
K_bar_fixed = 0.01
alpha = 0.6  # A common choice for alpha

print(f"Executando {num_realizations} simulacoes para Aproximacao Estocastica...")

for i in range(num_realizations):
    u, y = simulate_arx_system(N, theta_true, noise_variance)

    # b) Estimate parameters with stochastic approximation methods
    sa_grad_estimates[i, :] = estimate_stochastic_approx_grad(y, u, K_bar_fixed, alpha)
    sa_gn_estimates[i, :] = estimate_stochastic_approx_gn(y, u, alpha)

print("Simulacoes concluidas. Gerando novos histogramas...")

# --- Plotting Histograms for Stochastic Approximation ---
fig_b, axs_b = plt.subplots(2, 2, figsize=(14, 10))
fig_b.suptitle("Histogramas dos Estimadores de Aproximacao Estocastica")

# Top row: Stochastic Approx. Gradient-like
axs_b[0, 0].hist(sa_grad_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs_b[0, 0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs_b[0, 0].axvline(
    np.mean(sa_grad_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(sa_grad_estimates[:, 0]):.4f}",
)
axs_b[0, 0].set_title("Aprox. Estocastica (Gradiente): $\\hat{a}_1$")
axs_b[0, 0].set_xlabel("Valor Estimado de $a_1$")
axs_b[0, 0].set_ylabel("Frequencia")
axs_b[0, 0].legend()
axs_b[0, 0].grid(True)

axs_b[0, 1].hist(sa_grad_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs_b[0, 1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs_b[0, 1].axvline(
    np.mean(sa_grad_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(sa_grad_estimates[:, 1]):.4f}",
)
axs_b[0, 1].set_title("Aprox. Estocastica (Gradiente): $\\hat{b}_1$")
axs_b[0, 1].set_xlabel("Valor Estimado de $b_1$")
axs_b[0, 1].set_ylabel("Frequencia")
axs_b[0, 1].legend()
axs_b[0, 1].grid(True)

# Bottom row: Stochastic Approx. Gauss-Newton-like
axs_b[1, 0].hist(sa_gn_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs_b[1, 0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs_b[1, 0].axvline(
    np.mean(sa_gn_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(sa_gn_estimates[:, 0]):.4f}",
)
axs_b[1, 0].set_title("Aprox. Estocastica (Gauss-Newton): $\\hat{a}_1$")
axs_b[1, 0].set_xlabel("Valor Estimado de $a_1$")
axs_b[1, 0].set_ylabel("Frequencia")
axs_b[1, 0].legend()
axs_b[1, 0].grid(True)

axs_b[1, 1].hist(sa_gn_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs_b[1, 1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs_b[1, 1].axvline(
    np.mean(sa_gn_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(sa_gn_estimates[:, 1]):.4f}",
)
axs_b[1, 1].set_title("Aprox. Estocastica (Gauss-Newton): $\\hat{b}_1$")
axs_b[1, 1].set_xlabel("Valor Estimado de $b_1$")
axs_b[1, 1].set_ylabel("Frequencia")
axs_b[1, 1].legend()
axs_b[1, 1].grid(True)

fig_b.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1b_histograma.png")
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Minimum Covariance Estimator (Kalman Filter) Implementation
# --------------------------------------------------------------------------
def estimate_min_covariance(y, u, R):
    """
    Estimates parameters using the recursive minimum covariance estimator.

    Args:
        y (np.array): Output signal.
        u (np.array): Input signal.
        R (float): Measurement noise covariance.

    Returns:
        np.array: Final estimated parameter vector.
    """
    num_params = 2

    # Initialize parameter estimates and covariance matrix
    theta_est = np.zeros(num_params)

    # High initial uncertainty
    P = np.identity(num_params) * 1000

    N = len(y)

    for k in range(1, N):
        # Form the regressor vector
        psi = np.array([y[k - 1], u[k - 1]])

        # 1. Calculate the optimal gain K_k
        # The term psi.T @ P @ psi + R is a scalar
        denominator = psi.T @ P @ psi + R
        Kk = (P @ psi) / denominator

        # 2. Update parameter estimates
        error = y[k] - psi.T @ theta_est
        theta_est = theta_est + Kk * error

        # 3. Update the covariance matrix
        P = P - np.outer(Kk, psi.T @ P)

    return theta_est


# Store final estimates
min_cov_estimates = np.zeros((num_realizations, 2))

print(
    f"Executando {num_realizations} simulacoes para o Estimador de Minima Covariancia..."
)

for i in range(num_realizations):
    u, y = simulate_arx_system(N, theta_true, noise_variance)

    # c) Estimate parameters with minimum covariance estimator
    min_cov_estimates[i, :] = estimate_min_covariance(y, u, noise_variance)

print("Simulacoes concluidas. Gerando histogramas...")

# --- Plotting Histograms for Minimum Covariance Estimator ---
fig_c, axs_c = plt.subplots(1, 2, figsize=(14, 5))
fig_c.suptitle("Histogramas do Estimador de Minima Covariancia")

# Histogram for a1
axs_c[0].hist(min_cov_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs_c[0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs_c[0].axvline(
    np.mean(min_cov_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(min_cov_estimates[:, 0]):.4f}",
)
axs_c[0].set_title("Minima Covariancia: $\\hat{a}_1$")
axs_c[0].set_xlabel("Valor Estimado de $a_1$")
axs_c[0].set_ylabel("Frequencia")
axs_c[0].legend()
axs_c[0].grid(True)

# Histogram for b1
axs_c[1].hist(min_cov_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs_c[1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs_c[1].axvline(
    np.mean(min_cov_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(min_cov_estimates[:, 1]):.4f}",
)
axs_c[1].set_title("Minima Covariancia: $\\hat{b}_1$")
axs_c[1].set_xlabel("Valor Estimado de $b_1$")
axs_c[1].set_ylabel("Frequencia")
axs_c[1].legend()
axs_c[1].grid(True)

fig_c.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1c_histograma.png")
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Recursive Least Squares (RLS) Estimator Implementation
# --------------------------------------------------------------------------
def estimate_rls(y, u):
    """
    Estimates parameters using the Recursive Least Squares (RLS) estimator.

    Args:
        y (np.array): Output signal.
        u (np.array): Input signal.

    Returns:
        np.array: Final estimated parameter vector.
    """
    num_params = 2

    # Initialize parameter estimates and covariance matrix
    theta_est = np.zeros(num_params)

    # High initial uncertainty
    P = np.identity(num_params) * 1000

    N = len(y)

    for k in range(1, N):
        # Form the regressor vector
        psi = np.array([y[k - 1], u[k - 1]])

        # 1. Calculate the RLS gain K_k
        # Note the denominator is (1 + psi.T @ P @ psi)
        denominator = 1 + psi.T @ P @ psi
        Kk = (P @ psi) / denominator

        # 2. Update parameter estimates
        error = y[k] - psi.T @ theta_est
        theta_est = theta_est + Kk * error

        # 3. Update the covariance matrix
        P = P - np.outer(Kk, psi.T @ P)

    return theta_est


# Store final estimates
rls_estimates = np.zeros((num_realizations, 2))

print(
    f"Executando {num_realizations} simulacoes para o Estimador de Minimos Quadrados Recursivo..."
)

for i in range(num_realizations):
    u, y = simulate_arx_system(N, theta_true, noise_variance)

    # d) Estimate parameters with RLS estimator
    rls_estimates[i, :] = estimate_rls(y, u)

print("Simulacoes concluidas. Gerando histogramas...")

# --- Plotting Histograms for RLS Estimator ---
fig_d, axs_d = plt.subplots(1, 2, figsize=(14, 5))
fig_d.suptitle("Histogramas do Estimador de Minimos Quadrados Recursivo (MQR)")

# Histogram for a1
axs_d[0].hist(rls_estimates[:, 0], bins=15, edgecolor="black", alpha=0.7)
axs_d[0].axvline(
    a1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {a1_true}",
)
axs_d[0].axvline(
    np.mean(rls_estimates[:, 0]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(rls_estimates[:, 0]):.4f}",
)
axs_d[0].set_title("MQR: $\\hat{a}_1$")
axs_d[0].set_xlabel("Valor Estimado de $a_1$")
axs_d[0].set_ylabel("Frequencia")
axs_d[0].legend()
axs_d[0].grid(True)

# Histogram for b1
axs_d[1].hist(rls_estimates[:, 1], bins=15, edgecolor="black", alpha=0.7)
axs_d[1].axvline(
    b1_true,
    color="r",
    linestyle="--",
    linewidth=2,
    label=f"Valor Verdadeiro = {b1_true}",
)
axs_d[1].axvline(
    np.mean(rls_estimates[:, 1]),
    color="g",
    linestyle="-",
    linewidth=2,
    label=f"Media Estimada = {np.mean(rls_estimates[:, 1]):.4f}",
)
axs_d[1].set_title("MQR: $\\hat{b}_1$")
axs_d[1].set_xlabel("Valor Estimado de $b_1$")
axs_d[1].set_ylabel("Frequencia")
axs_d[1].legend()
axs_d[1].grid(True)

fig_d.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1d_histograma.png")
plt.show()

In [ ]:
# Compute bias, variance and covariance matrix euclidian norm for all estimators
def compute_statistics(estimates, true_params):
    """
    Computes bias, variance, and covariance matrix euclidian norm for given estimates.

    Args:
        estimates (np.array): Array of shape (num_realizations, num_params) with parameter estimates.
        true_params (np.array): True parameter vector.

    Returns:
        dict: A dictionary containing bias, variance, and covariance matrix euclidian norm.
    """
    bias = np.mean(estimates, axis=0) - true_params
    variance = np.var(estimates, axis=0)
    covariance_norm = np.linalg.norm(np.cov(estimates, rowvar=False))

    return {"bias": bias, "variance": variance, "covariance_norm": covariance_norm}


# Gather statistics for each estimator
estimators_stats = {
    "Gradiente": compute_statistics(grad_estimates, theta_true),
    "Gauss-Newton": compute_statistics(gn_estimates, theta_true),
    "Aprox. Estocastica (Gradiente)": compute_statistics(sa_grad_estimates, theta_true),
    "Aprox. Estocastica (Gauss-Newton)": compute_statistics(
        sa_gn_estimates, theta_true
    ),
    "Minima Covariancia": compute_statistics(min_cov_estimates, theta_true),
    "MQR": compute_statistics(rls_estimates, theta_true),
}

# Print the statistics
for name, stats in estimators_stats.items():
    print(f"Estatisticas para {name}:")
    print(f"\tVies (a1, b1): [{stats['bias'][0]:.2g}, {stats['bias'][1]:.2g}]")
    print(
        f"\tVariancia (a1, b1): [{stats['variance'][0]:.2g}, {stats['variance'][1]:.2g}]"
    )
    print(f"\tNorma da Matriz de Covariancia: {stats['covariance_norm']:.2g}")

In [ ]:
# --------------------------------------------------------------------------
# Analysis of the Impact of Data Length (N) on Statistics
# --------------------------------------------------------------------------


# --- Helper Function to Compute Statistics ---
def compute_estimator_statistics(estimates, true_params):
    """
    Computes bias and variance for given estimates.

    Args:
        estimates (np.array): Array of shape (num_realizations, num_params) with parameter estimates.
        true_params (np.array): True parameter vector.

    Returns:
        dict: A dictionary containing bias vector and variance vector.
    """

    bias_vector = np.mean(estimates, axis=0) - true_params
    variance_vector = np.var(estimates, axis=0)

    return {"bias": bias_vector, "variance": variance_vector}


# --- Analysis Configuration ---
N_values = [2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
num_realizations = 100  # Keep the number of realizations constant

# Consider all estimators
estimator_functions = {
    "Gradiente": lambda y, u: estimate_gradient_like(y, u, K_bar_fixed),
    "Gauss-Newton": lambda y, u: estimate_gauss_newton_like(y, u),
    "Aprox. Estocastica (Gradiente)": lambda y, u: estimate_stochastic_approx_grad(
        y, u, K_bar_fixed, alpha
    ),
    "Aprox. Estocastica (GN)": lambda y, u: estimate_stochastic_approx_gn(y, u, alpha),
    "Minima Covariancia": lambda y, u: estimate_min_covariance(y, u, noise_variance),
    "MQR": lambda y, u: estimate_rls(y, u),
}

# Data structure to store the results from the simulation loops
results = {name: {"bias_norm": [], "mean_variance": []} for name in estimator_functions}

print("Analisando o impacto do numero de amostras (N) sobre as estatisticas...")

# --- Main Simulation Loop ---
for N_test in N_values:
    print(f"  Executando {num_realizations} simulacoes para N = {N_test}...")

    # Temporary storage for the estimates at the current N
    current_estimates = {
        name: np.zeros((num_realizations, 2)) for name in estimator_functions
    }

    for i in range(num_realizations):
        # Simulate the system with N_test samples
        u, y = simulate_arx_system(N_test, theta_true, noise_variance)

        # Run each estimator
        for name, func in estimator_functions.items():
            current_estimates[name][i, :] = func(y, u)

    # Calculate and store statistics for each estimator after all realizations
    for name in estimator_functions:
        stats = compute_estimator_statistics(current_estimates[name], theta_true)

        # Calculate scalar metrics for plotting
        bias_norm = np.linalg.norm(stats["bias"])
        mean_variance = np.mean(stats["variance"])

        # Append results for the current N
        results[name]["bias_norm"].append(bias_norm)
        results[name]["mean_variance"].append(mean_variance)

print("Analise concluida. Gerando graficos comparativos...")

# --- Plotting the Comparative Statistics ---
fig_e2, axs_e2 = plt.subplots(1, 2, figsize=(22, 7))
fig_e2.suptitle("Metricas Estatisticas dos Estimadores vs. Numero de Amostras (N)")

# Plot 1: Bias Norm
for name, data in results.items():
    axs_e2[0].plot(N_values, data["bias_norm"], "o-", label=name)
axs_e2[0].set_title("Norma do Vetor de Vies")
axs_e2[0].set_xlabel("Numero de Amostras (N)")
axs_e2[0].set_ylabel("Norma do Vies")
axs_e2[0].set_xscale("log")
axs_e2[0].set_yscale("log")
axs_e2[0].legend()
axs_e2[0].grid(True, which="both", ls="--")

# Plot 2: Mean Variance
for name, data in results.items():
    axs_e2[1].plot(N_values, data["mean_variance"], "o-", label=name)
axs_e2[1].set_title("Variancia Media das Estimativas")
axs_e2[1].set_xlabel("Numero de Amostras (N)")
axs_e2[1].set_ylabel("Variancia Media")
axs_e2[1].set_xscale("log")
axs_e2[1].set_yscale("log")
axs_e2[1].legend()
axs_e2[1].grid(True, which="both", ls="--")

plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1e_estatisticas_vs_N.png")
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Parameter Trajectory Analysis
# --------------------------------------------------------------------------

# --- Estimator Functions Modified to Return History ---


def estimate_gradient_like_history(y, u, K_bar):
    """Gradient-like estimator that returns the full history of estimates."""
    theta_est = np.zeros(2)
    # History array to store estimates at each step k
    history = np.zeros((len(y), 2))
    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])
        error = y[k] - psi.T @ theta_est
        Kk = K_bar * psi
        theta_est = theta_est + Kk * error
        history[k, :] = theta_est
    return history


def estimate_gauss_newton_like_history(y, u):
    """Gauss-Newton-like estimator that returns the full history of estimates."""
    theta_est = np.zeros(2)
    history = np.zeros((len(y), 2))
    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])
        psi_norm_sq = psi.T @ psi
        if psi_norm_sq < 1e-9:
            history[k, :] = theta_est  # Keep previous estimate
            continue
        error = y[k] - psi.T @ theta_est
        Kk = (1 / psi_norm_sq) * psi
        theta_est = theta_est + Kk * error
        history[k, :] = theta_est
    return history


def estimate_stochastic_approx_grad_history(y, u, K_bar, alpha):
    """Stochastic approximation (gradient-like) estimator that returns history."""
    theta_est = np.zeros(2)
    history = np.zeros((len(y), 2))
    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])
        error = y[k] - psi.T @ theta_est
        Kk = K_bar * (k ** (-alpha)) * psi
        theta_est = theta_est + Kk * error
        history[k, :] = theta_est
    return history


def estimate_stochastic_approx_gn_history(y, u, alpha):
    """Stochastic approximation (GN-like) estimator that returns history."""
    theta_est = np.zeros(2)
    history = np.zeros((len(y), 2))
    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])
        psi_norm_sq = psi.T @ psi
        if psi_norm_sq < 1e-9:
            history[k, :] = theta_est  # Keep previous estimate
            continue
        error = y[k] - psi.T @ theta_est
        Kk = (k ** (-alpha)) * (1 / psi_norm_sq) * psi
        theta_est = theta_est + Kk * error
        history[k, :] = theta_est
    return history


def estimate_min_covariance_history(y, u, R):
    """Minimum covariance estimator that returns the full history of estimates."""
    theta_est = np.zeros(2)
    P = np.identity(2) * 1000
    history = np.zeros((len(y), 2))
    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])
        denominator = psi.T @ P @ psi + R
        Kk = (P @ psi) / denominator
        error = y[k] - psi.T @ theta_est
        theta_est = theta_est + Kk * error
        P = P - np.outer(Kk, psi.T @ P)
        history[k, :] = theta_est
    return history


def estimate_rls_history(y, u):
    """RLS estimator that returns the full history of estimates."""
    theta_est = np.zeros(2)
    P = np.identity(2) * 1000
    history = np.zeros((len(y), 2))
    # Use N=1000 for a clear trajectory view
    N_traj = 1000
    for k in range(1, N_traj):
        psi = np.array([y[k - 1], u[k - 1]])
        denominator = 1 + psi.T @ P @ psi
        Kk = (P @ psi) / denominator
        error = y[k] - psi.T @ theta_est
        theta_est = theta_est + Kk * error
        P = P - np.outer(Kk, psi.T @ P)
        history[k, :] = theta_est
    return history


# Fix random seed for reproducibility (a single, fixed noise realization)
np.random.seed(0)

# Use N=1000 for a clear trajectory view
N_trajectory = 1000

print("Executando uma unica simulacao para analise da trajetoria...")

# Simulate the system once
u_traj, y_traj = simulate_arx_system(N_trajectory, theta_true, noise_variance)

# Get parameter history from each estimator
hist_grad = estimate_gradient_like_history(y_traj, u_traj, K_bar_fixed)
hist_gauss_newton = estimate_gauss_newton_like_history(y_traj, u_traj)
hist_sa_grad = estimate_stochastic_approx_grad_history(
    y_traj, u_traj, K_bar_fixed, alpha
)
hist_sa_gn = estimate_stochastic_approx_gn_history(y_traj, u_traj, alpha)
hist_min_cov = estimate_min_covariance_history(y_traj, u_traj, noise_variance)
hist_rls = estimate_rls_history(y_traj, u_traj)

# --- Plotting the Trajectories ---
fig_f, axs_f = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
fig_f.suptitle("Evolucao Temporal dos Parametros Estimados")

# Plot for a1
axs_f[0].plot(hist_grad[:, 0], label="Gradiente", alpha=0.8, lw=1.5)
axs_f[0].plot(hist_gauss_newton[:, 0], label="Gauss-Newton", alpha=0.2, lw=1.5)
axs_f[0].plot(
    hist_sa_grad[:, 0], label="Aprox. Estocastica (Gradiente)", alpha=0.8, lw=1.5
)
axs_f[0].plot(hist_sa_gn[:, 0], label="Aprox. Estocastica (GN)", alpha=0.8, lw=1.5)
axs_f[0].plot(hist_min_cov[:, 0], label="Minima Covariancia", lw=2.5)
axs_f[0].plot(hist_rls[:, 0], label="MQR", linestyle=":", lw=2.5)
axs_f[0].axhline(
    a1_true, color="r", linestyle="--", label=f"Valor Verdadeiro $a_1$ = {a1_true}"
)
axs_f[0].set_title("Estimativa de $\\hat{a}_1(k)$")
axs_f[0].set_ylabel("Valor Estimado")
axs_f[0].legend()
axs_f[0].set_ylim(
    np.min(
        [
            hist_grad[:, 0].min(),
            hist_sa_grad[:, 0].min(),
            hist_min_cov[:, 0].min(),
            hist_rls[:, 0].min(),
        ]
    )
    - 0.1,
    np.max(
        [
            hist_grad[:, 0].max(),
            hist_sa_grad[:, 0].max(),
            hist_min_cov[:, 0].max(),
            hist_rls[:, 0].max(),
        ]
    )
    + 0.1,
)

# Plot for b1
axs_f[1].plot(hist_grad[:, 1], label="Gradiente", alpha=0.8, lw=1.5)
axs_f[1].plot(hist_gauss_newton[:, 1], label="Gauss-Newton", alpha=0.2, lw=1.5)
axs_f[1].plot(
    hist_sa_grad[:, 1], label="Aprox. Estocastica (Gradiente)", alpha=0.8, lw=1.5
)
axs_f[1].plot(hist_sa_gn[:, 1], label="Aprox. Estocastica (GN)", alpha=0.8, lw=1.5)
axs_f[1].plot(hist_min_cov[:, 1], label="Minima Covariancia", lw=2.5)
axs_f[1].plot(hist_rls[:, 1], label="MQR", linestyle=":", lw=2.5)
axs_f[1].axhline(
    b1_true, color="r", linestyle="--", label=f"Valor Verdadeiro $b_1$ = {b1_true}"
)
axs_f[1].set_title("Estimativa de $\\hat{b}_1(k)$")
axs_f[1].set_xlabel("Amostra (k)")
axs_f[1].set_ylabel("Valor Estimado")
axs_f[1].legend()
axs_f[1].set_ylim(
    np.min(
        [
            hist_grad[:, 1].min(),
            hist_sa_grad[:, 1].min(),
            hist_min_cov[:, 1].min(),
            hist_rls[:, 1].min(),
        ]
    )
    - 0.1,
    np.max(
        [
            hist_grad[:, 1].max(),
            hist_sa_grad[:, 1].max(),
            hist_min_cov[:, 1].max(),
            hist_rls[:, 1].max(),
        ]
    )
    + 0.1,
)

plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_1f_trajetoria_parametros.png")
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Time-Varying Parameter Estimation
# --------------------------------------------------------------------------


# --- Function to Simulate the Time-Varying ARX System ---
def simulate_time_varying_arx(N, noise_var):
    """
    Simulates the ARX system with time-varying parameters:
    a1(k) = 0.707 * cos(0.01 * pi * k)
    b1(k) = 1 - a1(k)
    """
    # Initialize signals
    u = np.random.randn(N)
    eta = np.sqrt(noise_var) * np.random.randn(N)
    y = np.zeros(N)

    # Store true parameter evolution
    a1_true_hist = np.zeros(N)
    b1_true_hist = np.zeros(N)

    # Simulate the system dynamics
    for k in range(1, N):
        a1_k = 0.707 * np.cos(0.01 * np.pi * k)
        b1_k = 1 - a1_k
        a1_true_hist[k] = a1_k
        b1_true_hist[k] = b1_k

        y[k] = a1_k * y[k - 1] + b1_k * u[k - 1] + eta[k] + 0.5 * eta[k - 1]

    return u, y, a1_true_hist, b1_true_hist


# --- RLS with Forgetting Factor Estimator ---
def estimate_rls_forgetting_factor(y, u, forgetting_factor):
    """
    Estimates parameters using RLS with a forgetting factor (lambda).
    """
    num_params = 2
    theta_est = np.zeros(num_params)
    P = np.identity(num_params) * 1000

    history = np.zeros((len(y), num_params))

    for k in range(1, len(y)):
        psi = np.array([y[k - 1], u[k - 1]])

        # RLS-FF equations
        denominator = forgetting_factor + psi.T @ P @ psi
        Kk = (P @ psi) / denominator

        error = y[k] - psi.T @ theta_est
        theta_est = theta_est + Kk * error

        P = (1 / forgetting_factor) * (P - np.outer(Kk, psi.T @ P))

        history[k, :] = theta_est

    return history


# Simulation parameters
N_tv = 2000  # Number of samples for the time-varying case

# Generate data from the time-varying system
u_tv, y_tv, a1_true_tv, b1_true_tv = simulate_time_varying_arx(N_tv, noise_variance)

# Values for the forgetting factor to test
lambda_values = [0.999, 0.99, 0.95, 0.9]

# --- Plotting the results ---
fig_2a, axs_2a = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
fig_2a.suptitle("Rastreamento de Parametros Variantes no Tempo com RLS-FF")

# Plot evolution of a1
axs_2a[0].plot(a1_true_tv, "k--", label="Valor Verdadeiro ($a_1$)", linewidth=3)
for lam in lambda_values:
    hist = estimate_rls_forgetting_factor(y_tv, u_tv, lam)
    axs_2a[0].plot(hist[:, 0], label=f"$\\lambda = {lam}$", alpha=0.8)

axs_2a[0].set_title("Rastreamento do Parametro $\\hat{a}_1(k)$")
axs_2a[0].set_ylabel("Valor Estimado")
axs_2a[0].legend()
axs_2a[0].grid(True)
axs_2a[0].set_ylim(np.min(a1_true_tv) - 0.2, np.max(a1_true_tv) + 0.2)

# Plot evolution of b1
axs_2a[1].plot(b1_true_tv, "k--", label="Valor Verdadeiro ($b_1$)", linewidth=3)
for lam in lambda_values:
    hist = estimate_rls_forgetting_factor(y_tv, u_tv, lam)
    axs_2a[1].plot(hist[:, 1], label=f"$\\lambda = {lam}$", alpha=0.8)

axs_2a[1].set_title("Rastreamento do Parametro $\\hat{b}_1(k)$")
axs_2a[1].set_xlabel("Amostra (k)")
axs_2a[1].set_ylabel("Valor Estimado")
axs_2a[1].legend()
axs_2a[1].grid(True)
axs_2a[1].set_ylim(np.min(b1_true_tv), np.max(b1_true_tv) + 0.2)

plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig(output_dir + "3b_2a_rastreamento_lambda.png")
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# Effect of Persistence of Excitation
# --------------------------------------------------------------------------


# --- Function to Simulate with a Hold Input Signal ---
def simulate_with_hold_input(N, noise_var, hold_duration):
    """
    Simulates the time-varying system with an input that holds its
    value for a specified duration.
    """
    # Generate the hold input signal
    u = np.zeros(N)
    for k in range(0, N, hold_duration):
        # Generate a new random value and hold it
        val = np.random.randn()
        end_step = min(k + hold_duration, N)
        u[k:end_step] = val

    # Initialize other signals
    eta = np.sqrt(noise_var) * np.random.randn(N)
    y = np.zeros(N)
    a1_true_hist = np.zeros(N)
    b1_true_hist = np.zeros(N)

    # Simulate the system dynamics
    for k in range(1, N):
        a1_k = 0.707 * np.cos(0.01 * np.pi * k)
        b1_k = 1 - a1_k
        a1_true_hist[k] = a1_k
        b1_true_hist[k] = b1_k

        y[k] = a1_k * y[k - 1] + b1_k * u[k - 1] + eta[k] + 0.5 * eta[k - 1]

    return u, y, a1_true_hist, b1_true_hist


# Simulation parameters
N_pe = 2000  # PE for Persistence of Excitation

# Durations for which the input signal will be held constant
hold_durations = [1, 10, 25, 50]

# Choose a good forgetting factor from the previous analysis
lambda_fixed = 0.95

# Create subplots for each hold duration
fig_2b, axs_2b = plt.subplots(
    len(hold_durations), 2, figsize=(16, 4 * len(hold_durations)), sharex=True
)
fig_2b.suptitle(
    "Efeito da Persistencia de Excitacao no Rastreamento de Parametros", fontsize=16
)

for i, hold_d in enumerate(hold_durations):
    # Simulate system with the corresponding hold input
    u_pe, y_pe, a1_true_pe, b1_true_pe = simulate_with_hold_input(
        N_pe, noise_variance, hold_d
    )

    # Estimate parameters using RLS-FF
    hist_pe = estimate_rls_forgetting_factor(y_pe, u_pe, lambda_fixed)

    # --- Plotting for a1 ---
    ax_a1 = axs_2b[i, 0]
    ax_a1.plot(a1_true_pe, "k--", label="Valor Verdadeiro ($a_1$)", linewidth=2.5)
    ax_a1.plot(hist_pe[:, 0], label=f"Estimado (Hold={hold_d})", alpha=0.9)
    ax_a1.set_ylabel("Valor de $\\hat{a}_1(k)$")
    ax_a1.legend()
    ax_a1.grid(True)
    ax_a1.set_title(f"Entrada Constante por {hold_d} Amostras")
    ax_a1.set_ylim(np.min(a1_true_pe) - 0.2, np.max(a1_true_pe) + 0.2)

    # --- Plotting for b1 ---
    ax_b1 = axs_2b[i, 1]
    ax_b1.plot(b1_true_pe, "k--", label="Valor Verdadeiro ($b_1$)", linewidth=2.5)
    ax_b1.plot(hist_pe[:, 1], label=f"Estimado (Hold={hold_d})", alpha=0.9)
    ax_b1.set_ylabel("Valor de $\\hat{b}_1(k)$")
    ax_b1.legend()
    ax_b1.grid(True)
    ax_b1.set_title(f"Entrada Constante por {hold_d} Amostras")
    ax_b1.set_ylim(np.min(b1_true_pe), np.max(b1_true_pe) + 0.2)

# Common x-label
axs_2b[-1, 0].set_xlabel("Amostra (k)")
axs_2b[-1, 1].set_xlabel("Amostra (k)")

plt.tight_layout(rect=(0, 0.03, 1, 0.96))
plt.savefig(output_dir + "3b_2b_persistencia_excitacao.png")
plt.show()